<a href="https://colab.research.google.com/github/szabeebzaliz7-a11y/Datascience-And-Gen-AI/blob/main/Module_6_AI_and_Gen_AI_Module_End_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AI-powered paraphrasing tool

Develop an AI-powered paraphrasing tool as a Python application or module. The tool
should:

● Take a block of input text
● Use deep learning (preferably transformer models like T5, BERT, GPT, or similar
via Hugging Face Transformers)
● Generate a paraphrased version preserving meaning, improving clarity, and
ensuring originality.
● Include built-in checks for grammar, spelling, and fluency of output

In [6]:
!pip -q install transformers
!pip -q install sentence-transformers
!pip -q install language-tool-python
!pip -q install evaluate
!pip -q install rouge-score
!pip -q install sacrebleu
!pip -q install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.7 MB/s eta 0:00:00


In [7]:
import torch
import evaluate
import language_tool_python

from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

In [8]:
MODEL_NAME = "ramsrigouthamg/t5_paraphraser"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

grammar_tool = language_tool_python.LanguageTool("en-US")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

bleu = evaluate.load("bleu")

rouge = evaluate.load("rouge")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
import re
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------
# 1. Text Preprocessing
# ---------------------------
def preprocess(text):
    """
    Clean the input text by removing extra spaces.
    """
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# ---------------------------
# 2. Paraphrasing Function
# ---------------------------
def paraphrase(text):
    """
    Generate a paraphrased version of the input text
    using the pre-trained T5 paraphrasing model.
    """

    sentence = "paraphrase: " + text + " </s>"

    encoding = tokenizer(
        sentence,
        return_tensors="pt",
        max_length=256,
        truncation=True,
        padding="max_length"
    )

    outputs = model.generate(
        input_ids=encoding["input_ids"],
        attention_mask=encoding["attention_mask"],
        max_length=256,
        do_sample=True,
        top_k=120,
        top_p=0.95,
        temperature=1.2,
        early_stopping=True,
        num_return_sequences=1
    )

    paraphrased_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return paraphrased_text


# ---------------------------
# 3. Grammar Correction
# ---------------------------
def grammar_correct(text):
    """
    Correct grammar and spelling using LanguageTool.
    """

    matches = grammar_tool.check(text)

    corrected_text = language_tool_python.utils.correct(
        text,
        matches
    )

    return corrected_text


# ---------------------------
# 4. Evaluation Function
# ---------------------------
def evaluate_output(original, paraphrased):
    """
    Evaluate the generated paraphrase using:
    - BLEU Score
    - ROUGE-L Score
    - Semantic Similarity
    """

    bleu_score = bleu.compute(
        predictions=[paraphrased],
        references=[[original]]
    )

    rouge_score = rouge.compute(
        predictions=[paraphrased],
        references=[original]
    )

    original_embedding = embedding_model.encode([original])

    paraphrased_embedding = embedding_model.encode([paraphrased])

    similarity = cosine_similarity(
        original_embedding,
        paraphrased_embedding
    )[0][0]

    return {
        "BLEU Score": round(bleu_score["bleu"], 4),
        "ROUGE-L Score": round(rouge_score["rougeL"], 4),
        "Semantic Similarity": round(float(similarity), 4)
    }

In [10]:
# Get user input
text = input("Enter a paragraph to paraphrase:\n\n")

# Step 1: Preprocess
processed_text = preprocess(text)

# Step 2: Generate paraphrase
paraphrased_text = paraphrase(processed_text)

# Step 3: Grammar correction
final_output = grammar_correct(paraphrased_text)

# Step 4: Evaluation
scores = evaluate_output(processed_text, final_output)

# Display Results
print("\n" + "="*60)
print("ORIGINAL TEXT")
print("="*60)
print(processed_text)

print("\n" + "="*60)
print("PARAPHRASED TEXT")
print("="*60)
print(final_output)

print("\n" + "="*60)
print("EVALUATION METRICS")
print("="*60)

for metric, value in scores.items():
    print(f"{metric}: {value}")

Enter a paragraph to paraphrase:

my name is zabeeb zaizand i am studying in entri app as a course named datascience and gen ai 


[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



ORIGINAL TEXT
my name is zabeeb zaizand i am studying in entri app as a course named datascience and gen ai

PARAPHRASED TEXT
My name is Zagreb Wieland I am studying in entry app as a track named Data science and Gen Ai. And will be coming to India one day.

EVALUATION METRICS
BLEU Score: 0.0
ROUGE-L Score: 0.5957
Semantic Similarity: 0.7509
